# Electoral Systems Comparison Across Scenarios

Compares electoral systems across several real-world-grounded electorate scenarios,
measuring how close each system's outcome is to the **geometric median** of voter preferences.

## Setup

Voter preferences and candidate positions are represented as 2D vectors in $[0,1]^2$:
- **Dimension 1 (x-axis):** Economic axis — left $(0)$ to right $(1)$
- **Dimension 2 (y-axis):** Social axis — libertarian $(0)$ to authoritarian $(1)$

## Primary metric

For winner-take-all systems the outcome is the winning candidate's position.
For PR systems it is the **median legislator position** (50th percentile of the
seat-share distribution sorted along the economic axis) — consistent with the
pivot/median voter theorem applied to the legislature. The seat-share centroid is
also reported for reference.

The **distance to the geometric median** measures how far this outcome is from
the most central point in the voter distribution — the point minimising the sum
of Euclidean distances to all voters.

## Systems

**Winner-take-all:** Plurality (FPTP) · Two-Round Runoff · Instant Runoff (IRV) ·
Borda Count · Approval Voting · Score Voting · Condorcet (Schulze)

**Proportional:** Party-List PR (D'Hondt) · Mixed Member Proportional (MMP)

**Novel:** Fractional Ballot (σ=0.1) · Fractional Ballot (σ=0.3) · Fractional Ballot (σ=1.0)


---
## 0. Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Repo root on path ─────────────────────────────────────────────────────────
REPO_ROOT = Path("../").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from electoral_sim.scenario import load_all_scenarios
from electoral_sim.ballots import BallotProfile
from electoral_sim.systems import get_all_systems
from electoral_sim.fractional import (
    FractionalBallotDiscrete,
    FractionalBallotContinuous,
    fractional_ballot_systems,
    sigma_sweep,
    weight_entropy_sweep,
    _mean_weights,
)
from electoral_sim.metrics import run_simulation, run_monte_carlo, summarize_monte_carlo
from electoral_sim.utils import (
    plot_electorate,
    plot_election_result,
    plot_all_systems_spatial,
    plot_metric_bar,
    plot_grouped_metrics,
    plot_scenario_heatmap,
    plot_radar,
    plot_monte_carlo_distributions,
)
from electoral_sim.utils.viz_metrics import METRIC_CONFIG, _short_name

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
RNG  = np.random.default_rng(SEED)

# ── Output directory for saved figures ────────────────────────────────────────
FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":        120,
    "font.family":       "sans-serif",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

print(f"Repo root : {REPO_ROOT}")
print(f"Figures   : {FIGURES_DIR}")


---
## 1. Load scenarios and run all systems

Load all scenario YAML files, derive ballot profiles under sincere voting,
run all systems on each scenario, and collect metrics.
The three Fractional Ballot variants (σ=0.1, 0.3, 1.0) are included alongside
the nine standard systems.


In [ ]:
SCENARIOS_DIR = REPO_ROOT / "configs"/ "scenarios" 

scenarios = load_all_scenarios(SCENARIOS_DIR, rng=RNG)

# Standard systems + both Fractional variants at σ = 0.1, 0.3, 1.0
base_systems       = get_all_systems(rng=RNG)
fractional_systems = fractional_ballot_systems(variant="both")   # Discrete + Continuous
systems            = base_systems + fractional_systems

# Separate handles for targeted analysis later
frac_discrete   = fractional_ballot_systems(variant="discrete")
frac_continuous = fractional_ballot_systems(variant="continuous")

print(f"Loaded {len(scenarios)} scenarios, {len(systems)} systems\n")
for cfg, e, c in scenarios:
    print(f"  {cfg['name']:35s}  {e.n_voters:,} voters  {c.n_candidates} candidates")
print()
print("Systems:")
for s in systems:
    tag = " ◆" if "Continuous" in s.name else (" ●" if "Discrete" in s.name else "")
    print(f"  {s.name}{tag}")
print("\n◆ = Fractional Continuous (weighted legislature)")
print("● = Fractional Discrete (single winner)")


In [ ]:
# Run every system on every scenario
# results_by_scenario : {scenario_name: [ElectionResult, ...]}
# metrics_by_scenario : {scenario_name: [ElectionMetrics, ...]}
# ballots_by_scenario : {scenario_name: BallotProfile}

results_by_scenario = {}
metrics_by_scenario = {}
ballots_by_scenario = {}

for cfg, e, c in scenarios:
    name    = cfg["name"]
    ballots = BallotProfile.from_preferences(e, c)
    results = [s.run(ballots, c) for s in systems]
    metrics = run_simulation(e, c, systems)

    results_by_scenario[name] = results
    metrics_by_scenario[name] = metrics
    ballots_by_scenario[name] = ballots

    geo_median = e.geometric_median()
    print(f"\n{'─'*72}")
    print(f"Scenario: {name}")
    print(f"  Geometric median : ({geo_median[0]:.3f}, {geo_median[1]:.3f})")
    print(f"  Mean preference  : ({e.mean()[0]:.3f}, {e.mean()[1]:.3f})")
    print(f"  {'System':<38} {'Outcome':>26}  {'d(median)':>9}  {'centroid_d':>10}")
    for r, m in zip(results, metrics):
        pos = r.outcome_position
        if r.is_pr:
            outcome_str  = f"({pos[0]:.3f},{pos[1]:.3f}) [med.leg]"
            centroid_str = f"{m.centroid_distance_to_median:.4f}"
        else:
            outcome_str  = f"({pos[0]:.3f},{pos[1]:.3f}) [winner]"
            centroid_str = "—"
        print(f"  {r.system_name:<38} {outcome_str:>26}  {m.distance_to_median:>9.4f}  {centroid_str:>10}")

---
## 2. Electorate portraits

Each panel shows the voter density (KDE contours), all candidate positions,
and the geometric median (teal ✛) and mean (red ◆) of the voter distribution.
This establishes the spatial context before looking at outcomes.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), dpi=120)
axes_flat = axes.flatten()

for ax, (cfg, e, c) in zip(axes_flat, scenarios):
    plot_electorate(
        e, c,
        title=f"{cfg['name']}\n({cfg['real_world_analog']})",
        ax=ax,
    )

axes_flat[-1].set_visible(False)

fig.suptitle("Electorate Portraits — Voter Density and Candidate Positions",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig01_electorate_portraits.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig01_electorate_portraits.png", bbox_inches="tight", dpi=300)
plt.show()

**Reading the plot.** The blue contours show where voters are concentrated.
The teal **✛** is the geometric median — the point minimising total distance to
all voters. The red **◆** is the arithmetic mean. In the unimodal scenario these
coincide; in the polarized and asymmetric scenarios they diverge noticeably.
The ideal electoral system should elect (or produce an outcome near) the teal marker.


---
## 3. Per-scenario spatial outcome maps

For each scenario one panel per electoral system. The ★ marks the winner;
for PR systems the ✕ marks the seat-share-weighted centroid (reference only) and
the purple ◆ marks the median legislator position (primary outcome metric).
Distance to geometric median is annotated in each panel corner.


In [ ]:
for cfg, e, c in scenarios:
    name    = cfg["name"]
    results = results_by_scenario[name]

    fig = plot_all_systems_spatial(
        e, c, results,
        n_cols=4,
        suptitle=f"Scenario: {name}  ({cfg['real_world_analog']})",
        figsize_per_panel=(4.5, 4.2),
    )

    slug = name.lower().replace(" ", "_").replace("/", "_")
    fig.savefig(FIGURES_DIR / f"fig02_{slug}_spatial.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURES_DIR / f"fig02_{slug}_spatial.png", bbox_inches="tight", dpi=300)
    plt.show()

---
## 4. Single-metric ranking per scenario

Horizontal bar charts rank systems by **distance to geometric median** within each scenario.
Lower is better. The best-performing system for each scenario has a black border.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11), dpi=120)
axes_flat = axes.flatten()

for ax, (cfg, e, c) in zip(axes_flat, scenarios):
    name    = cfg["name"]
    metrics = metrics_by_scenario[name]
    plot_metric_bar(
        metrics,
        metric="distance_to_median",
        title=cfg["name"],
        ax=ax,
    )

axes_flat[-1].set_visible(False)
fig.suptitle("Distance to Geometric Median — All Scenarios",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig03_metric_bar_per_scenario.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig03_metric_bar_per_scenario.png", bbox_inches="tight", dpi=300)
plt.show()

---
## 5. Cross-scenario heatmap

The central comparative figure. Rows are electoral systems, columns are scenarios.
Color encodes distance to the geometric median (green = closer = better).
Black borders highlight the best system in each column (scenario).

This plot reveals which systems are **consistently good** across all scenarios versus
which are **situationally best** (good in some contexts, poor in others).


In [ ]:
SHORT_NAMES = {
    "Unimodal Consensus":            "Consensus",
    "Polarized Bimodal":             "Polarized",
    "Multimodal Fragmented":         "Fragmented",
    "Dominant Party with Minorities":"Dominant",
    "Asymmetric Skewed":             "Skewed",
}

scenario_summaries_short = {
    SHORT_NAMES.get(k, k): v
    for k, v in metrics_by_scenario.items()
}

fig = plot_scenario_heatmap(
    scenario_summaries_short,
    metric="distance_to_median",
    title="Distance to Geometric Median — Electoral Systems × Scenarios",
    figsize=(13, 6),
)
fig.savefig(FIGURES_DIR / "fig04_heatmap_d_median.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig04_heatmap_d_median.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Same heatmap for majority satisfaction
fig = plot_scenario_heatmap(
    scenario_summaries_short,
    metric="majority_satisfaction",
    title="Majority Satisfaction — Electoral Systems × Scenarios",
    figsize=(13, 6),
)
fig.savefig(FIGURES_DIR / "fig04b_heatmap_majority_sat.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig04b_heatmap_majority_sat.png", bbox_inches="tight", dpi=300)
plt.show()

---
## 6. All-metrics comparison per scenario

The grouped bar chart shows all six metrics simultaneously for each scenario.
Values are **min-max normalised within each metric** and oriented so that
a shorter bar is always better, regardless of the metric's natural direction.


In [ ]:
for cfg, e, c in scenarios:
    name    = cfg["name"]
    metrics = metrics_by_scenario[name]

    fig = plot_grouped_metrics(
        metrics,
        title=f"All Metrics — {name}  ({cfg['real_world_analog']})",
        normalize=True,
        figsize=(14, 5),
    )
    slug = name.lower().replace(" ", "_").replace("/", "_")
    fig.savefig(FIGURES_DIR / f"fig05_{slug}_all_metrics.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURES_DIR / f"fig05_{slug}_all_metrics.png", bbox_inches="tight", dpi=300)
    plt.show()

---
## 7. Summary statistics table

Distance-to-median for every system × scenario combination.
The best system per scenario is highlighted in green.
The rank table (sorted by mean rank) identifies the most consistently good systems.


In [ ]:
sys_names  = [_short_name(s.name) for s in systems]
scen_names = [SHORT_NAMES.get(cfg["name"], cfg["name"]) for cfg, _, _ in scenarios]

data = {}
for (cfg, e, c), scen_short in zip(scenarios, scen_names):
    metrics = metrics_by_scenario[cfg["name"]]
    data[scen_short] = [round(m.distance_to_median, 4) for m in metrics]

df = pd.DataFrame(data, index=sys_names)

def highlight_min(s):
    is_min = s == s.min()
    return ["background-color: #b7e4c7; font-weight: bold" if v else "" for v in is_min]

df.style \
  .apply(highlight_min, axis=0) \
  .set_caption("Distance to Geometric Median (lower = better)") \
  .format("{:.4f}")

In [ ]:
# Rank of each system within each scenario + mean rank across scenarios
rank_df = df.rank(axis=0, ascending=True).astype(int)
rank_df["Mean rank"] = df.rank(axis=0, ascending=True).mean(axis=1).round(2)
rank_df = rank_df.sort_values("Mean rank")

def highlight_rank1(s):
    return ["background-color: #b7e4c7; font-weight: bold" if v == 1 else "" for v in s]

rank_df.style \
  .apply(highlight_rank1, axis=0, subset=scen_names) \
  .set_caption("Rank by distance to geometric median (1 = best). Sorted by mean rank.")

---
## 8. Key observations

Structured summary of findings extracted directly from the simulation results.


In [ ]:
print("=" * 70)
print("KEY FINDINGS — Distance to Geometric Median")
print("=" * 70)
print("  For PR systems: d(median) uses the median legislator position.")
print("  centroid_d is shown for reference (tends to be near voter mean).")

for (cfg, e, c), scen_short in zip(scenarios, scen_names):
    name    = cfg["name"]
    metrics = metrics_by_scenario[name]

    sorted_m = sorted(metrics, key=lambda m: m.distance_to_median)
    best     = sorted_m[0]
    worst    = sorted_m[-1]

    geo_median     = e.geometric_median()
    mean_pref      = e.mean()
    mean_median_gap = np.linalg.norm(geo_median - mean_pref)

    print(f"\n{name} ({cfg['real_world_analog']})")
    print(f"  Mean–median gap      : {mean_median_gap:.4f}")
    print(f"  Best system          : {_short_name(best.system_name):<28} d={best.distance_to_median:.4f}")
    print(f"  Worst system         : {_short_name(worst.system_name):<28} d={worst.distance_to_median:.4f}")
    print(f"  Best/worst ratio     : {worst.distance_to_median / max(best.distance_to_median, 1e-9):.2f}x")

    plurality_m = next(m for m in metrics if "Plurality" in m.system_name)
    if plurality_m.distance_to_median > best.distance_to_median * 1.5:
        print(f"  ⚠  Plurality is {plurality_m.distance_to_median / max(best.distance_to_median, 1e-9):.1f}x worse than best")

    for m in metrics:
        if m.is_pr and not np.isnan(m.centroid_distance_to_median):
            gap = m.distance_to_median - m.centroid_distance_to_median
            print(f"  ℹ  {_short_name(m.system_name)}: centroid_d={m.centroid_distance_to_median:.4f}  "
                  f"med.leg_d={m.distance_to_median:.4f}  artifact_gap={gap:+.4f}")

print("\n" + "=" * 70)

---
## 9. Fractional Ballot deep-dive

The Fractional Ballot assigns each voter a weight vector over candidates via
a Boltzmann (softmax) kernel with temperature σ:

$$w_{ik} = \frac{e^{-d_{ik}/\sigma}}{\sum_j e^{-d_{ij}/\sigma}}$$

**Two variants:**

| Variant | Outcome | Use case |
|---------|---------|----------|
| **Discrete** | Nearest candidate to the centroid wins outright | Primaries, single-nominee elections |
| **Continuous** | `seat_shares = mean_w` — every candidate holds fractional power | Policy voting in a weighted legislature |

In the Continuous variant, on any policy bill candidate $k$ casts a vote
with weight $\bar{w}_k = \frac{1}{n}\sum_i w_{ik}$, so the effective
policy outcome is always a convex combination of all candidates' positions.

**Special cases (both variants):**
- σ → 0 : weights concentrate on nearest candidate → equivalent to Plurality
- σ → ∞ : weights become uniform → equal power for all candidates


In [ ]:
# ── Sigma frontier: spatial plot helper ──────────────────────────────────────
def plot_sigma_frontier(electorate, candidates, ax=None,
                        title="Fractional Ballot σ-Frontier", sigmas=None):
    from electoral_sim.utils.viz_electorate import (
        _kde_contours, _plot_candidates, _plot_reference_points, _style_spatial_ax,
    )
    from electoral_sim.fractional import FractionalBallotContinuous, sigma_sweep

    if sigmas is None:
        sigmas = np.logspace(-2, 0.7, 50).tolist()

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6, 5.5), dpi=120)
    else:
        fig = ax.get_figure()

    ballots = BallotProfile.from_preferences(electorate, candidates)
    sweep   = sigma_sweep(sigmas, ballots, candidates, variant="continuous")
    path    = np.array([pos for _, pos in sweep])

    _kde_contours(ax, electorate.preferences, alpha=0.40)
    _plot_candidates(ax, candidates)
    _plot_reference_points(ax, electorate)

    # σ-frontier path with plasma colormap (cool=low σ, warm=high σ)
    from matplotlib.collections import LineCollection
    points = path.reshape(-1, 1, 2)
    segs   = np.concatenate([points[:-1], points[1:]], axis=1)
    lc     = LineCollection(segs, cmap="plasma", linewidth=2.5, zorder=9)
    lc.set_array(np.linspace(0, 1, len(segs)))
    ax.add_collection(lc)

    # Mark σ = 0.1, 0.3, 1.0
    for s_mark in [0.1, 0.3, 1.0]:
        fb_res = FractionalBallotContinuous(sigma=s_mark).run(ballots, candidates)
        pos    = fb_res.outcome_position
        ax.scatter(pos[0], pos[1], s=90, c="#9b5de5", zorder=10,
                   edgecolors="white", linewidths=1.2)
        ax.annotate(f"σ={s_mark}", xy=pos, xytext=(5, 5),
                    textcoords="offset points", fontsize=8, color="#6a0dad")

    ax.annotate("σ→0 (Plurality)", xy=path[0], xytext=(-50, -15),
                textcoords="offset points", fontsize=7.5, color="#555",
                arrowprops=dict(arrowstyle="->", color="#888", lw=0.8))
    ax.annotate("σ→∞ (Uniform)", xy=path[-1], xytext=(8, 8),
                textcoords="offset points", fontsize=7.5, color="#555",
                arrowprops=dict(arrowstyle="->", color="#888", lw=0.8))

    _style_spatial_ax(ax, title, electorate.dim_names)
    if standalone:
        fig.tight_layout()
    return fig


In [ ]:
# ── Polarized Bimodal: Continuous vs Discrete + entropy ──────────────────────
FOCUS    = "Polarized Bimodal"
focus_cfg, focus_e, focus_c = next(
    (cfg, e, c) for cfg, e, c in scenarios if cfg["name"] == FOCUS
)
focus_results = results_by_scenario[FOCUS]
focus_metrics = metrics_by_scenario[FOCUS]
ballots_f     = ballots_by_scenario[FOCUS]

sigmas_dense = np.logspace(-2, 0.7, 60)
gm           = focus_e.geometric_median()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), dpi=120)

# ── Left: spatial σ-frontier ─────────────────────────────────────────────────
plot_sigma_frontier(focus_e, focus_c, ax=axes[0],
                    title="σ-Frontier (spatial) — Polarized Bimodal")

# ── Middle: d(median) vs σ for both variants ──────────────────────────────────
for variant, color, ls, label in [
    ("continuous", "#9b5de5", "-",  "Continuous (weighted legislature)"),
    ("discrete",   "#f4a261", "--", "Discrete (single winner)"),
]:
    sweep = sigma_sweep(sigmas_dense.tolist(), ballots_f, focus_c, variant=variant)
    dists = [np.linalg.norm(pos - gm) for _, pos in sweep]
    axes[1].plot(sigmas_dense, dists, color=color, linewidth=2.2,
                 linestyle=ls, zorder=5, label=label)

# Reference lines for selected standard systems
ref_systems = {
    "Plurality (FPTP)":    ("#e63946", ":"),
    "Borda Count":         ("#4361ee", ":"),
    "Condorcet (Schulze)": ("#2a9d8f", "-."),
}
for r, m in zip(focus_results, focus_metrics):
    if r.system_name in ref_systems:
        col, ls = ref_systems[r.system_name]
        axes[1].axhline(m.distance_to_median, color=col, linewidth=1.2,
                        linestyle=ls, alpha=0.75,
                        label=f"{_short_name(r.system_name)}  d={m.distance_to_median:.4f}")

axes[1].set_xscale("log")
axes[1].set_xlabel("σ (temperature)", fontsize=10)
axes[1].set_ylabel("Distance to geometric median", fontsize=10)
axes[1].set_title("d(median) vs σ — Continuous vs Discrete", fontsize=10, fontweight="bold")
axes[1].legend(fontsize=7.5, loc="upper right", framealpha=0.88)
axes[1].grid(alpha=0.25)

# ── Right: weight entropy vs σ ────────────────────────────────────────────────
entropy_sweep = weight_entropy_sweep(sigmas_dense.tolist(), ballots_f, focus_c)
sigmas_e, entropies = zip(*entropy_sweep)
axes[2].plot(sigmas_e, entropies, color="#2a9d8f", linewidth=2.2)

# Mark σ = 0.1, 0.3, 1.0
for s_mark in [0.1, 0.3, 1.0]:
    mean_w  = _mean_weights(ballots_f.distances, s_mark)
    entropy = float(-np.sum(mean_w * np.log(mean_w + 1e-12)))
    axes[2].scatter([s_mark], [entropy], s=80, c="#2a9d8f", zorder=7,
                    edgecolors="white", linewidths=1.0)
    axes[2].annotate(f"σ={s_mark}", (s_mark, entropy),
                     xytext=(5, 4), textcoords="offset points", fontsize=8)

axes[2].axhline(np.log(focus_c.n_candidates), color="gray", linewidth=1.0,
                linestyle="--", alpha=0.7,
                label=f"Max entropy (uniform, H={np.log(focus_c.n_candidates):.2f})")
axes[2].set_xscale("log")
axes[2].set_xlabel("σ (temperature)", fontsize=10)
axes[2].set_ylabel("Entropy of mean weight vector H(w̄)", fontsize=10)
axes[2].set_title("Power concentration vs σ", fontsize=10, fontweight="bold")
axes[2].legend(fontsize=8, framealpha=0.88)
axes[2].grid(alpha=0.25)

fig.suptitle("Fractional Ballot — Polarized Bimodal Deep-Dive",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig06_fractional_deepdive_polarized.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig06_fractional_deepdive_polarized.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# ── Weight distribution per candidate across σ values (Continuous variant) ───
# Shows how legislative power is distributed as σ varies.
# Each grouped bar = one candidate; each color = one σ value.

sigmas_show = [0.1, 0.3, 1.0]
colors_sigma = ["#e63946", "#9b5de5", "#2a9d8f"]

fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 4.5), dpi=120)

for ax, (cfg, e, c) in zip(axes, scenarios):
    ballots_ = BallotProfile.from_preferences(e, c)
    n_cands  = c.n_candidates
    x        = np.arange(n_cands)
    width    = 0.22

    for j, (s, col) in enumerate(zip(sigmas_show, colors_sigma)):
        mean_w = _mean_weights(ballots_.distances, s)
        offset = (j - 1) * width
        bars   = ax.bar(x + offset, mean_w, width=width * 0.92,
                        color=col, alpha=0.85, label=f"σ={s}")

    ax.set_xticks(x)
    ax.set_xticklabels(c.labels, rotation=35, ha="right", fontsize=7.5)
    ax.set_ylabel("Mean weight w̄ₖ (legislative power share)", fontsize=8)
    ax.set_title(cfg["name"], fontsize=9, fontweight="bold")
    ax.axhline(1 / n_cands, color="gray", linewidth=0.8, linestyle="--",
               alpha=0.6, label="Uniform (1/K)")
    ax.set_ylim(0, None)
    ax.grid(axis="y", alpha=0.25)
    if ax == axes[0]:
        ax.legend(fontsize=7.5, framealpha=0.88)

fig.suptitle(
    "Fractional Ballot Continuous — Legislative Weight Distribution per Candidate\n"
    "(dashed line = uniform equal shares; σ↑ → flatter distribution)",
    fontsize=11, fontweight="bold",
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig07_fractional_weight_distribution.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig07_fractional_weight_distribution.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# ── Sigma frontier across all scenarios ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10), dpi=120)
axes_flat = axes.flatten()

for ax, (cfg, e, c) in zip(axes_flat, scenarios):
    plot_sigma_frontier(
        e, c, ax=ax,
        title=f"{cfg['name']}",
    )

axes_flat[-1].set_visible(False)
fig.suptitle("Fractional Ballot σ-Frontier — All Scenarios",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig07_sigma_frontier_all_scenarios.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig07_sigma_frontier_all_scenarios.png", bbox_inches="tight", dpi=300)
plt.show()

---
## 10. Scenario deep-dive: Polarized Bimodal

The polarized scenario is the most diagnostically interesting because the geometric
median falls in a **low-density gap** between two voter clusters. This is exactly
where systems diverge most — some are captured by the larger cluster, others find
the central position.


In [ ]:
geo_med = focus_e.geometric_median()
mean_p  = focus_e.mean()
print(f"Geometric median : ({geo_med[0]:.3f}, {geo_med[1]:.3f})")
print(f"Mean preference  : ({mean_p[0]:.3f}, {mean_p[1]:.3f})")
print(f"Gap (Euclidean)  : {np.linalg.norm(geo_med - mean_p):.4f}")
print()

sorted_pairs = sorted(
    zip(focus_results, focus_metrics),
    key=lambda x: x[1].distance_to_median
)

print(f"  {'System':<32} {'outcome':>22}  {'d_median':>9}  {'winner(s)'}")
for r, m in sorted_pairs:
    pos           = r.outcome_position
    winner_labels = [focus_c.labels[i] for i in r.winner_indices]
    centroid_note = ""
    if r.is_pr:
        c_pos = r.centroid_position
        centroid_note = f"  centroid=({c_pos[0]:.3f},{c_pos[1]:.3f}) d={m.centroid_distance_to_median:.4f}"
    print(f"  {_short_name(r.system_name):<32} ({pos[0]:.3f},{pos[1]:.3f})  {m.distance_to_median:>9.4f}"
          f"  {', '.join(winner_labels)}{centroid_note}")

In [ ]:
# Best 3 and worst 3 systems in the polarized scenario
top3    = [r for r, m in sorted_pairs[:3]]
bottom3 = [r for r, m in sorted_pairs[-3:]]

for group_label, group_results in [("Best 3 systems", top3), ("Worst 3 systems", bottom3)]:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), dpi=120)
    for ax, r in zip(axes, group_results):
        plot_election_result(focus_e, focus_c, r, ax=ax)
    fig.suptitle(f"Polarized Bimodal — {group_label}",
                 fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    slug = group_label.lower().replace(" ", "_")
    fig.savefig(FIGURES_DIR / f"fig08_polarized_{slug}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURES_DIR / f"fig08_polarized_{slug}.png", bbox_inches="tight", dpi=300)
    plt.show()

---
## 11. Scenario deep-dive: Multimodal Fragmented

The fragmented scenario tests how systems handle a genuinely multi-party electorate
with five distinct voter clusters. PR systems are expected to excel here; Plurality
is expected to fail badly by electing the plurality cluster leader.


In [ ]:
FOCUS2 = "Multimodal Fragmented"
focus2_cfg, focus2_e, focus2_c = next((cfg, e, c) for cfg, e, c in scenarios if cfg["name"] == FOCUS2)
focus2_results = results_by_scenario[FOCUS2]
focus2_metrics = metrics_by_scenario[FOCUS2]

sorted_pairs2 = sorted(
    zip(focus2_results, focus2_metrics),
    key=lambda x: x[1].distance_to_median
)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), dpi=120)
for ax, (r, m) in zip(axes, sorted_pairs2[:3]):
    plot_election_result(focus2_e, focus2_c, r, ax=ax)

fig.suptitle("Multimodal Fragmented — Top 3 Systems",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig09_fragmented_top3.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "fig09_fragmented_top3.png", bbox_inches="tight", dpi=300)
plt.show()

print("\nPR vs. winner-take-all — Fragmented scenario:")
print(f"  {'System':<38} {'d(median)':>10}  {'majority_sat':>14}")
for r, m in sorted_pairs2:
    tag = " [PR]" if r.is_pr else ""
    print(f"  {_short_name(r.system_name) + tag:<38} {m.distance_to_median:>10.4f}  {m.majority_satisfaction:>14.4f}")

---
## 12. Monte Carlo stability analysis

Single-run results can be sensitive to the specific random draw of the electorate.
Here we run 200 Monte Carlo trials per scenario — each trial re-samples voters from
the same distribution — and show the distribution of distance-to-median for each system.

Systems with **low variance** are robust; systems with **high variance** are sensitive
to the specific realisation of the electorate.


In [ ]:
import copy
from electoral_sim.electorate import from_config as electorate_from_config
from electoral_sim.candidates import from_config as candidates_from_config

MC_TRIALS = 200
MC_VOTERS = 2000   # smaller for speed; results are stable

mc_results_by_scenario = {}

for cfg, e, c in scenarios:
    name = cfg["name"]

    mc_cfg = copy.deepcopy(cfg)
    mc_cfg["n_voters"] = MC_VOTERS

    def make_factory(cfg_):
        def factory(rng):
            e_ = electorate_from_config(cfg_, rng=rng)
            c_ = candidates_from_config(cfg_)
            return e_, c_
        return factory

    mc_results = run_monte_carlo(
        make_factory(mc_cfg),
        systems,
        n_trials=MC_TRIALS,
        rng=np.random.default_rng(SEED),
    )
    mc_results_by_scenario[name] = mc_results
    print(f"  {name}: {MC_TRIALS} trials done")

print("Monte Carlo complete.")

In [ ]:
for cfg, e, c in scenarios:
    name = cfg["name"]
    mc   = mc_results_by_scenario[name]

    fig = plot_monte_carlo_distributions(
        mc,
        metric="distance_to_median",
        title=f"MC Stability ({MC_TRIALS} trials) — {name}",
        figsize=(13, 5),
    )
    slug = name.lower().replace(" ", "_").replace("/", "_")
    fig.savefig(FIGURES_DIR / f"fig10_{slug}_mc_stability.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURES_DIR / f"fig10_{slug}_mc_stability.png", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# MC summary table: mean ± std of distance_to_median, all scenarios
rows = []
for cfg, _, _ in scenarios:
    scen_short = SHORT_NAMES.get(cfg["name"], cfg["name"])
    summary    = summarize_monte_carlo(mc_results_by_scenario[cfg["name"]])
    for sys_name, stats in summary.items():
        rows.append({
            "Scenario":      scen_short,
            "System":        _short_name(sys_name),
            "Mean d_median": round(stats["distance_to_median_mean"], 4),
            "Std d_median":  round(stats["distance_to_median_std"],  4),
        })

mc_df = pd.DataFrame(rows)
pivot = mc_df.pivot_table(index="System", columns="Scenario", values="Mean d_median")
pivot["Overall mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("Overall mean")

pivot.style \
  .background_gradient(cmap="RdYlGn_r", axis=None) \
  .format("{:.4f}") \
  .set_caption(f"Monte Carlo mean distance to geometric median ({MC_TRIALS} trials). Lower = better.")